# Lesson 5 : Foundry Tools

In Lesson 4, we have learned basic tool's usage by using built-in hosted tools in Agent Framework.  
As I have mentioned in Lesson 1, ```AzureAIClient``` is based on ```azure-ai-projects``` v2 SDK. You, therefore, can also integrate with a wide variety of Microsoft Foundry tools (such as, Azure AI Search, Microsoft SharePoint, Microsoft Fabric, etc), when you're using ```AzureAIClient```.

> Note : For available Foundry tools, see [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/tool-catalog) or [API reference](https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models?view=azure-python-preview).

In this exercise, we simply change the example in Lesson 4 to use "Grounding with Bing Search" tool.

> Note : When you need to provide the credential or identity (such as, OAuth access token) in tools, process authentication or authorization in your apps (such as, show the login UI, process OAuth flow, etc) and get the identity for tool's usage. (Some tools - such as, SharePoint tool, Fabric tool, etc - handle the identity.)

## 1. Create and connect to Bing Grounding Search resource

Before writing code, please create and connect to "Grounding with Bing Search" resource  in Microsoft Foundry as follows.

1. Open [Azure Portal](https://portal.azure.com) and create a new "Grounding with Bing Search" resource.
2. Open Foundry Portal (new portal), click "Operate" menu, and select "Admin" in left-side navigation.
3. Select your project, and connect to above "Grounding with Bing Search" resource by clicking "Add connection".

## 2. Run with "Grounding with Bing Search" tool

First we create a ```AzureAIClient``` object.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we build an agent with "Grounding with Bing Search" tool's configuration as follows.

This configuration is the same as the configuration used by [REST API](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/bing-tools?view=foundry&tabs=grounding-with-bing&pivots=rest) in ```azure-ai-projects```.

The connection string for Bing Grounding search is the following format, and **please change connection string to your settings**. (You can also get connection string in Foundry Portal.)

```/subscriptions/{subscription-id}/resourceGroups/{resource-name}/providers/Microsoft.CognitiveServices/accounts/{foundry-resource-name}/projects/{foundry-project-name}/connections/{connected-resource-name}```

> Note : For security reasons, set connection string in environment variable (such as, ```BING_PROJECT_CONNECTION_ID```) and use it for ```project_connection_id``` property in production.

In [2]:
from agent_framework import HostedWebSearchTool

# ToDo : fill your settings
# (or retrieve BING_PROJECT_CONNECTION_ID from Foundry Portal)
AZURE_SUBSCRIPTION_ID = "xxxxxxxxxxxxxxxxxxxxx"
RESOURCE_GROUP_NAME = "xxxxxxxxxxxxxxxxxxxxx"
FOUNDRY_RESOURCE_NAME = "xxxxxxxxxxxxxxxxxxxxx"
FOUNDRY_PROJECT_NAME = "xxxxxxxxxxxxxxxxxxxxx"
CONNECTED_RESOURCE_NAME = "xxxxxxxxxxxxxxxxxxxxx"

BING_PROJECT_CONNECTION_ID = f"/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME}"

agent = client.create_agent(
    name="WeatherAgentWithBingSearch",
    instructions="あなたは、気象情報に関する Agent です。",  # "you are an agent about weather information."
    tools=[
        {
            "type": "bing_grounding",
            "bing_grounding": {
                "search_configurations": [
                    {
                        "project_connection_id": BING_PROJECT_CONNECTION_ID
                    }
                ]
            }
        }
    ],
)

Now we run the agent as follows.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("今日の大阪の天気と気温を教えてください。")  # "tell me the weather and temperature in Osaka today."
display(Markdown(result.text))

今日（1/13）の大阪（大阪市）**の天気は「晴れ」**、**最高 13℃／最低 3℃**の予報です。【5:1†source】